# Build the FULL SemanticKITTI dataset for MRA / RaCA and upload it to Kaggle (private datasets)

Same method as our `building-subset` notebook (same downloads, same MobileStereoNet depth, same compact format),
but with **every labelled frame** of sequences 00-10 instead of 400 + 150. The result is too big for Kaggle's 20 GB
notebook output, so it is built in `/tmp` **one group of sequences at a time**, and every finished group is uploaded
as its own **private Kaggle dataset** `<you>/semantickitti-mra-full-<group>` (about 15-20 GB each):

| group | sequences | labelled frames (approx.) | use |
|---|---|---|---|
| **D** | 08 (val) + 09 | 815 val + 318 train | built first: also gives the full validation set |
| **A** | 00, 01, 04 | 1,182 train | |
| **B** | 02, 03 | 1,092 train | |
| **C** | 05, 06, 07, 10 | 1,242 train | |
| T | 04 only | 55 train | **quick test** (~20-30 min) to check the whole pipeline first |

**Before running**
1. Right panel: Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons -> Secrets: add **`KAGGLE_API_TOKEN`** (kaggle.com -> Settings -> API -> Generate New Token) and switch it on
   for this notebook. (Older key pair also works: secrets `KAGGLE_USERNAME` + `KAGGLE_KEY`.)
3. Nothing to edit: it builds all groups D, A, B, C (check `KAGGLE_USER` in the settings cell is your username).
4. **Save Version -> Save & Run All (Commit)**. It runs in the background (you can close the browser).
   Groups that already exist on Kaggle are skipped, so if the 12 h limit stops a run, just commit again.

Progress lines: `images .../N`, `sequence XX: depth`, `converted ...`, `UPLOADED <dataset>`.

In [ ]:
# ---- Settings ----
KAGGLE_USER = "daniyalimran"              # your Kaggle username (datasets are created under it, private)
GROUPS      = ["D", "A", "B", "C"]        # whole dataset (D first = validation). Quick pipeline test only: ["T"]
PREFIX      = "semantickitti-mra-full"    # dataset slug prefix -> <user>/<prefix>-<group>

GROUP_SEQS = {"T": ["04"], "D": ["08", "09"], "A": ["00", "01", "04"], "B": ["02", "03"], "C": ["05", "06", "07", "10"]}

import os
os.environ.update({"MRA_DIR": "/kaggle/working/MRA_TIP", "TMP_DIR": "/tmp/kitti_build", "N_WORKERS": "16"})
print("groups:", {g: GROUP_SEQS[g] for g in GROUPS}, "| datasets:", [f"{KAGGLE_USER}/{PREFIX}-{g.lower()}" for g in GROUPS])

## A. Install MRA's software (same cells as building-subset; skipped if already installed)

In [ ]:
%%bash
# 1. Check GPU and free disk space
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
df -h /tmp /kaggle/working | sed 1d

In [ ]:
%%bash
if /tmp/mra_env/bin/python -c "import mmdet3d, spconv, torch_scatter" 2>/dev/null; then echo "Already installed in this session - skipping"; exit 0; fi
# 2. Install micromamba (a small conda) and create a Python 3.8 environment
set -e
cd /tmp
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
/tmp/bin/micromamba create -y -q -p /tmp/mra_env -c conda-forge python=3.8 pip > /tmp/mamba.log 2>&1 \
  || { tail -40 /tmp/mamba.log; echo FAILED; exit 1; }
/tmp/mra_env/bin/python --version && echo "OK: Python 3.8 environment ready" 

In [ ]:
%%bash
if /tmp/mra_env/bin/python -c "import mmdet3d, spconv, torch_scatter" 2>/dev/null; then echo "Already installed in this session - skipping"; exit 0; fi
# 3. Pin versions that the old packages need (stops pip from pulling newer, incompatible ones)
cat > /tmp/constraints.txt <<'EOF'
numpy==1.19.5
setuptools==59.5.0
yapf==0.40.1
torch==1.9.1+cu111
torchvision==0.10.1+cu111
EOF
PIP="/tmp/mra_env/bin/python -m pip"
$PIP install -q --upgrade "pip<24" > /tmp/pip0.log 2>&1 && $PIP install -q setuptools==59.5.0 wheel ninja >> /tmp/pip0.log 2>&1 \
  && echo "OK: pip ready" || { tail -30 /tmp/pip0.log; echo FAILED; }

In [ ]:
%%bash
if /tmp/mra_env/bin/python -c "import mmdet3d, spconv, torch_scatter" 2>/dev/null; then echo "Already installed in this session - skipping"; exit 0; fi
# 4. PyTorch 1.9.1 for CUDA 11.1 (about 2 GB download)
PIP="/tmp/mra_env/bin/python -m pip"
$PIP install -q -c /tmp/constraints.txt torch==1.9.1+cu111 torchvision==0.10.1+cu111 \
  -f https://download.pytorch.org/whl/torch_stable.html > /tmp/pip1.log 2>&1 \
  || { tail -30 /tmp/pip1.log; echo FAILED; exit 1; }
/tmp/mra_env/bin/python -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0)); x=torch.randn(512,512,device='cuda'); print('GPU matmul OK:', float((x@x).sum())==float((x@x).sum()))" \
  && echo "OK: PyTorch works on the GPU" || echo FAILED

In [ ]:
%%bash
if /tmp/mra_env/bin/python -c "import mmdet3d, spconv, torch_scatter" 2>/dev/null; then echo "Already installed in this session - skipping"; exit 0; fi
# 5. (spconv 2.3.6, not MRA's 2.1.25: 2.1.25's GPU sort crashes on real-size inputs; same weight layout)
# mmcv-full 1.4.0 (prebuilt), mmdet 2.14.0, mmseg 0.14.1, spconv, torch-scatter, torchmetrics, timm
PIP="/tmp/mra_env/bin/python -m pip"
$PIP install -q -c /tmp/constraints.txt mmcv-full==1.4.0 \
  -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.9.0/index.html > /tmp/pip2.log 2>&1 \
  && $PIP install -q -c /tmp/constraints.txt mmdet==2.14.0 mmsegmentation==0.14.1 >> /tmp/pip2.log 2>&1 \
  && $PIP install -q -c /tmp/constraints.txt spconv-cu113==2.3.6 >> /tmp/pip2.log 2>&1 \
  && $PIP install -q -c /tmp/constraints.txt torch-scatter==2.0.8 -f https://data.pyg.org/whl/torch-1.9.0+cu111.html >> /tmp/pip2.log 2>&1 \
  && $PIP install -q -c /tmp/constraints.txt torchmetrics==0.11.4 timm==0.6.13 gdown >> /tmp/pip2.log 2>&1 \
  && echo "OK: mmcv / mmdet / mmseg / spconv / torch-scatter installed" \
  || { tail -40 /tmp/pip2.log; echo FAILED; }

In [ ]:
%%bash
if /tmp/mra_env/bin/python -c "import mmdet3d, spconv, torch_scatter" 2>/dev/null; then echo "Already installed in this session - skipping"; exit 0; fi
# 6. mmdet3d v0.17.1 from source, WITHOUT compiling its CUDA operators.
#    MRA only uses mmdet3d's registries and base classes. Three operators always need a
#    CUDA 11.1 compiler, so we skip all 11 and put placeholder files in their place.
PIP="/tmp/mra_env/bin/python -m pip"
cd /tmp && rm -rf mmdetection3d
git clone -q https://github.com/open-mmlab/mmdetection3d.git && cd mmdetection3d && git checkout -q v0.17.1
cat > /tmp/patch_mmdet3d.py <<'EOF'
# Make mmdet3d v0.17.1 installable without compiling its CUDA operators.
# MRA does not use these operators; placeholder modules keep `import mmdet3d` working.
import os, re, sys
root = sys.argv[1] if len(sys.argv) > 1 else "."
setup_path = os.path.join(root, "setup.py")
src = open(setup_path).read()
pairs = re.findall(r"name='(\w+)',\s*module='([\w.]+)'", src)
if "ext_modules=[] and [" not in src:
    src = src.replace("ext_modules=[", "ext_modules=[] and [", 1)
    open(setup_path, "w").write(src)
STUB = '''# Placeholder: this compiled operator was not built (MRA does not use it).
def __getattr__(name):
    def _not_built(*args, **kwargs):
        raise NotImplementedError(__name__ + "." + name + " was not compiled")
    return _not_built
'''
for name, module in pairs:
    path = os.path.join(root, *module.split("."), name + ".py")
    open(path, "w").write(STUB)
    print("placeholder:", module + "." + name)
print("patched setup.py; extensions skipped:", len(pairs))
EOF
/tmp/mra_env/bin/python /tmp/patch_mmdet3d.py /tmp/mmdetection3d | tail -1
# old numba import path -> current one (only occurrence in mmdet3d)
sed -i 's/from numba.errors import/from numba.core.errors import/' mmdet3d/datasets/pipelines/data_augment_utils.py
$PIP install -q -c /tmp/constraints.txt numba==0.53.0 networkx==2.2 plyfile trimesh==2.35.39 scikit-image==0.19.3 \
  nuscenes-devkit==1.1.9 lyft_dataset_sdk==0.0.8 tensorboard==2.11.2 > /tmp/pip3.log 2>&1 \
  || { tail -40 /tmp/pip3.log; echo "FAILED (mmdet3d dependencies)"; exit 1; }
$PIP install -v -e . --no-deps --no-build-isolation > /tmp/mmdet3d_build.log 2>&1 \
  || { tail -60 /tmp/mmdet3d_build.log; echo "FAILED (mmdet3d build)"; exit 1; }
cd /tmp && /tmp/mra_env/bin/python -c "import mmdet3d, mmdet3d.models, mmdet3d.datasets; print('mmdet3d', mmdet3d.__version__)" \
  && echo "OK: mmdet3d installed" || echo FAILED

In [ ]:
%%bash
# 7. Get MRA's code, the ImageNet ResNet-50 weights it starts from, and SGN-T's trained weights
cd /kaggle/working
[ -d MRA_TIP ] || git clone -q https://github.com/PKU-ICST-MIPL/MRA_TIP.git
cd MRA_TIP && mkdir -p ckpts
[ -f ckpts/resnet50-19c8e357.pth ] || wget -q https://download.pytorch.org/models/resnet50-19c8e357.pth -O ckpts/resnet50-19c8e357.pth
ls -la ckpts/resnet50-19c8e357.pth && echo "OK: ResNet-50 weights"
# SGN-T, SemanticKITTI (from SGN's model zoo on Google Drive)
[ -f ckpts/sgn-T.pth ] || /tmp/mra_env/bin/gdown -q 1cDKX6SpyoAHr-cfmjTRDsNF-3_JgtH3p -O ckpts/sgn-T.pth
ls -la ckpts/sgn-T.pth && echo "OK: SGN-T weights" || echo "FAILED: could not download SGN-T weights (Google Drive may be rate-limiting; we can retry later)" 

## B. Tools: Kaggle command-line tool + credentials, build scripts

In [ ]:
# Kaggle CLI (current version understands KAGGLE_API_TOKEN), remotezip, and the credentials from Kaggle Secrets
import subprocess, os
subprocess.run("pip install -q -U kaggle remotezip", shell=True, check=True)
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
def _secret(name):
    try:
        return _s.get_secret(name)
    except Exception:
        return None
_tok = _secret("KAGGLE_API_TOKEN")
if _tok:
    os.environ["KAGGLE_API_TOKEN"] = _tok
else:
    _u, _k = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
    assert _u and _k, "Add the secret KAGGLE_API_TOKEN (Add-ons -> Secrets) and switch it on for this notebook"
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = _u, _k
r = subprocess.run(["kaggle", "datasets", "list", "--mine", "-v"], capture_output=True, text=True)
assert r.returncode == 0, "Kaggle credentials rejected:\n" + r.stdout + r.stderr
print("OK: Kaggle CLI works with your credentials")
os.makedirs("/kaggle/working/tools", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/tools/build_full.py
"""Build SemanticKITTI for MRA (full mode: every labelled frame of a group of sequences), without
downloading the 65 GB image zip. Based on build_subset.py; new: phase selectgroup and ONLY_SEQ for images.

Phases (run in order):  stats  select  meta  labels  images
Output layout matches what MRA's SemanticKittiDataset reads:
  OUT/sequences/XX/{calib.txt, poses.txt, image_2/*.png, voxels/*.bin}
  OUT/labels/XX/{fid}_1_1.npy (uint8 here; expand_subset.py restores MRA's float32), {fid}_1_2.npy
  OUT/sequences_msnet3d_lidar/sequences/XX/*.bin   (made later by depth_to_lidar.py)
"""
import os, re, sys, json, zipfile, random, subprocess, time
import numpy as np
import yaml

MRA = os.environ.get("MRA_DIR", "/kaggle/working/MRA_TIP")
OUT = os.environ.get("OUT_DIR", "/kaggle/working/kitti_subset/dataset")
TMP = os.environ.get("TMP_DIR", "/tmp/kitti_build")
N_TRAIN = int(os.environ.get("N_TRAIN", 400))
N_VAL = int(os.environ.get("N_VAL", 150))
MIN_RARE = int(os.environ.get("MIN_RARE", 20))   # voxels of rare classes for a frame to count as "rare"
SEED = 0

TRAIN_SEQS = ["00", "01", "02", "03", "04", "05", "06", "07", "09", "10"]
VAL_SEQ = "08"
OFFSETS = [-12, -9, -6, -3]                      # MRA-T temporal frames
RARE = {2: "bicycle", 3: "motorcycle", 6: "person", 7: "bicyclist", 8: "motorcyclist"}

VOXELS_URL = "http://www.semantic-kitti.org/assets/data_odometry_voxels.zip"
LABELS_URL = "http://www.semantic-kitti.org/assets/data_odometry_labels.zip"
COLOR_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_color.zip"
CALIB_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_calib.zip"
KPOSES_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_poses.zip"

VOX_RE = re.compile(r"sequences/(\d\d)/voxels/(\d{6})\.(label|invalid|bin)$")


# ---------------------------------------------------------------- helpers
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def remap_lut(yaml_path):
    """Same as MRA's io_data._get_remap_lut: SemanticKITTI ids -> 20 SSC classes, 0 = empty, 255 = ignore."""
    cfg = yaml.safe_load(open(yaml_path))
    lm = cfg["learning_map"]
    lut = np.zeros(max(lm.keys()) + 100, dtype=np.int32)
    lut[list(lm.keys())] = list(lm.values())
    lut[lut == 0] = 255
    lut[0] = 0
    return lut


def unpack(compressed):
    """Same as MRA's io_data.unpack (bit-packed flags -> one uint8 per voxel)."""
    return np.unpackbits(compressed)  # MSB first, identical to the shift-based version


def decode_label(label_bytes, invalid_bytes, lut):
    """Return the 256x256x32 float32 label exactly as MRA's label_preprocess.py writes *_1_1.npy."""
    label = np.frombuffer(label_bytes, dtype=np.uint16)
    invalid = unpack(np.frombuffer(invalid_bytes, dtype=np.uint8))
    label = lut[label.astype(np.uint16)].astype(np.float32)
    label[np.isclose(invalid, 1)] = 255
    return label.reshape(256, 256, 32)


def downsample_1_2(label):
    """Vectorised version of MRA's _downsample_label(label, (256,256,32), 2). Returns uint8 (128,128,16)."""
    b = label.astype(np.int64).reshape(128, 2, 128, 2, 16, 2).transpose(0, 2, 4, 1, 3, 5).reshape(128, 128, 16, 8)
    c0 = (b == 0).sum(-1)
    c255 = (b == 255).sum(-1)
    empty = (c0 + c255) > 0.95 * 8
    counts = np.stack([(b == k).sum(-1) for k in range(1, 20)], -1)   # classes 1..19
    major = counts.argmax(-1) + 1                                       # ties -> smallest class, like np.bincount().argmax()
    return np.where(empty, np.where(c0 > c255, 0, 255), major).astype(np.uint8)


def read_poses_lines(text):
    return [l for l in text.strip().splitlines() if l.strip()]


def wget(url, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(dst) < 1000:
        log("downloading", url)
        subprocess.check_call(["wget", "-q", "-c", "--timeout=60", "--tries=10", "-O", dst, url])
    return dst


# ---------------------------------------------------------------- phases
def phase_stats():
    """Per-frame class counts for every labelled frame of the train and val sequences."""
    zpath = wget(VOXELS_URL, f"{TMP}/voxels.zip")
    lut = remap_lut(f"{MRA}/preprocess/label/semantic-kitti.yaml")
    zf = zipfile.ZipFile(zpath)
    members = {}
    for n in zf.namelist():
        m = VOX_RE.search(n)
        if m:
            members[(m.group(1), m.group(2), m.group(3))] = n
    rows = []
    seqs = TRAIN_SEQS + [VAL_SEQ]
    frames = sorted({(s, f) for (s, f, e) in members if s in seqs and e == "label"})
    log(f"labelled frames in train+val: {len(frames)}")
    for i, (s, f) in enumerate(frames):
        lab = decode_label(zf.read(members[(s, f, "label")]), zf.read(members[(s, f, "invalid")]), lut)
        cnt = np.bincount(lab[lab != 255].astype(np.int64).ravel(), minlength=20)[:20]
        occ = lab[(lab > 0) & (lab < 255)]
        far = lab[192:][(lab[192:] > 0) & (lab[192:] < 255)]          # forward 38.4-51.2 m
        rows.append({"seq": s, "frame": f, "counts": cnt.tolist(), "occupied": int(occ.size),
                     "far_occupied": int(far.size), "rare": int(sum(cnt[k] for k in RARE))})
        if i % 500 == 0:
            log(f"  stats {i}/{len(frames)}")
    os.makedirs(os.path.dirname(OUT), exist_ok=True)
    json.dump(rows, open(f"{os.path.dirname(OUT)}/frame_stats.json", "w"))
    log("stats done")


def phase_select():
    rows = json.load(open(f"{os.path.dirname(OUT)}/frame_stats.json"))
    rng = random.Random(SEED)

    def pick(cands, n, rare_share=0.5, spread=False):
        rare = [r for r in cands if r["rare"] >= MIN_RARE]
        n_r = min(int(n * rare_share), len(rare))
        chosen = rng.sample(rare, n_r)
        keys = {(r["seq"], r["frame"]) for r in chosen}
        rest = [r for r in cands if (r["seq"], r["frame"]) not in keys]
        k = min(n - n_r, len(rest))
        if spread:   # evenly spaced along the sequence -> representative of the whole drive
            idx = np.linspace(0, len(rest) - 1, k).round().astype(int) if k else []
            chosen += [rest[i] for i in idx]
        else:
            chosen += rng.sample(rest, k)
        return sorted(chosen, key=lambda r: (r["seq"], r["frame"]))

    train = pick([r for r in rows if r["seq"] in TRAIN_SEQS], N_TRAIN)
    val = pick([r for r in rows if r["seq"] == VAL_SEQ], N_VAL, spread=True)

    def summary(sel, pool):
        c = np.sum([r["counts"] for r in sel], 0); p = np.sum([r["counts"] for r in pool], 0)
        return {"frames": len(sel), "pool_frames": len(pool),
                "frames_with_rare": sum(r["rare"] >= MIN_RARE for r in sel),
                "rare_voxels": {RARE[k]: int(c[k]) for k in RARE},
                "pool_rare_voxels": {RARE[k]: int(p[k]) for k in RARE}}

    sel = {"train": [[r["seq"], r["frame"]] for r in train], "val": [[r["seq"], r["frame"]] for r in val],
           "summary": {"train": summary(train, [r for r in rows if r["seq"] in TRAIN_SEQS]),
                       "val": summary(val, [r for r in rows if r["seq"] == VAL_SEQ])},
           "settings": {"N_TRAIN": N_TRAIN, "N_VAL": N_VAL, "MIN_RARE": MIN_RARE, "SEED": SEED}}
    json.dump(sel, open(f"{os.path.dirname(OUT)}/selection.json", "w"), indent=1)
    log(json.dumps(sel["summary"], indent=1))


def phase_meta():
    """calib.txt and poses.txt for every train/val sequence (MRA loads poses for all of them)."""
    from remotezip import RemoteZip
    seqs = TRAIN_SEQS + [VAL_SEQ]
    with RemoteZip(CALIB_URL, timeout=(30, 120)) as rz:
        for n in rz.namelist():
            m = re.search(r"sequences/(\d\d)/calib\.txt$", n)
            if m and m.group(1) in seqs:
                d = f"{OUT}/sequences/{m.group(1)}"; os.makedirs(d, exist_ok=True)
                open(f"{d}/calib.txt", "wb").write(rz.read(n))
    # poses: SemanticKITTI's poses.txt (voxels zip, else labels zip), else KITTI ground-truth poses
    found = {}
    zf = zipfile.ZipFile(f"{TMP}/voxels.zip")
    for n in zf.namelist():
        m = re.search(r"sequences/(\d\d)/poses\.txt$", n)
        if m and m.group(1) in seqs:
            found[m.group(1)] = zf.read(n)
    if len(found) < len(seqs):
        try:
            with RemoteZip(LABELS_URL, timeout=(30, 120)) as rz:
                for n in rz.namelist():
                    m = re.search(r"sequences/(\d\d)/poses\.txt$", n)
                    if m and m.group(1) in seqs and m.group(1) not in found:
                        found[m.group(1)] = rz.read(n)
        except Exception as e:
            log("labels zip not readable:", repr(e))
    src = "SemanticKITTI"
    if len(found) < len(seqs):
        src = "KITTI ground truth (fallback)"
        with RemoteZip(KPOSES_URL, timeout=(30, 120)) as rz:
            for n in rz.namelist():
                m = re.search(r"poses/(\d\d)\.txt$", n)
                if m and m.group(1) in seqs and m.group(1) not in found:
                    found[m.group(1)] = rz.read(n)
    for s in seqs:
        open(f"{OUT}/sequences/{s}/poses.txt", "wb").write(found[s])
    log(f"calib + poses written for {len(seqs)} sequences (poses from {src})")


def needed_frames(sel):
    """Every image frame MRA will read, mapped to the labelled frame that uses it (frames are never shared)."""
    ref_of = {}
    for s, f in sel["train"] + sel["val"]:
        n_poses = len(read_poses_lines(open(f"{OUT}/sequences/{s}/poses.txt").read()))
        fid = int(f)
        ref_of.setdefault(s, {})[f] = f
        for o in OFFSETS:
            t = fid + o
            tid = f if (t < 0 or t > n_poses - 1) else str(t).zfill(6)
            ref_of[s][tid] = f
    return ref_of


def phase_labels():
    sel = json.load(open(f"{os.path.dirname(OUT)}/selection.json"))
    lut = remap_lut(f"{MRA}/preprocess/label/semantic-kitti.yaml")
    zf = zipfile.ZipFile(f"{TMP}/voxels.zip")
    members = {}
    for n in zf.namelist():
        m = VOX_RE.search(n)
        if m:
            members[(m.group(1), m.group(2), m.group(3))] = n
    pairs = sel["train"] + sel["val"]
    for i, (s, f) in enumerate(pairs):
        vd = f"{OUT}/sequences/{s}/voxels"; ld = f"{OUT}/labels/{s}"
        os.makedirs(vd, exist_ok=True); os.makedirs(ld, exist_ok=True)
        open(f"{vd}/{f}.bin", "wb").write(zf.read(members[(s, f, "bin")]))
        lab = decode_label(zf.read(members[(s, f, "label")]), zf.read(members[(s, f, "invalid")]), lut)
        np.save(f"{ld}/{f}_1_1.npy", lab.astype(np.uint8))   # compact; values are 0-19 or 255. expand_subset.py restores float32
        np.save(f"{ld}/{f}_1_2.npy", downsample_1_2(lab))
        if i % 100 == 0:
            log(f"  labels {i}/{len(pairs)}")
    log(f"labels written for {len(pairs)} frames")


def phase_images():
    from remotezip import RemoteZip
    from concurrent.futures import ThreadPoolExecutor
    sel = json.load(open(f"{os.path.dirname(OUT)}/selection.json"))
    ref_of = needed_frames(sel)
    only = os.environ.get("ONLY_SEQ")                    # full mode: one sequence at a time (disk)
    if only:
        ref_of = {s: v for s, v in ref_of.items() if s == only}
    json.dump(ref_of, open(f"{TMP}/ref_of.json", "w"))
    jobs = []
    for s, frames in ref_of.items():
        os.makedirs(f"{OUT}/sequences/{s}/image_2", exist_ok=True)
        os.makedirs(f"{TMP}/stereo/sequences/{s}/image_3", exist_ok=True)
        link = f"{TMP}/stereo/sequences/{s}/image_2"
        if not os.path.exists(link):
            os.symlink(f"{OUT}/sequences/{s}/image_2", link)
        for t in frames:
            jobs.append((f"dataset/sequences/{s}/image_2/{t}.png", f"{OUT}/sequences/{s}/image_2/{t}.png"))
            jobs.append((f"dataset/sequences/{s}/image_3/{t}.png", f"{TMP}/stereo/sequences/{s}/image_3/{t}.png"))
    import threading
    def missing():
        return [j for j in jobs if not (os.path.exists(j[1]) and os.path.getsize(j[1]) > 0)]
    lock = threading.Lock(); progress = {"n": 0}

    def worker(chunk):
        rz = None
        for src, dst in chunk:
            for attempt in range(4):
                try:
                    if rz is None:
                        rz = RemoteZip(COLOR_URL, timeout=(30, 120))   # without a timeout a stalled download waits forever
                    data = rz.read(src)
                    with open(dst + ".part", "wb") as fh:
                        fh.write(data)
                    os.replace(dst + ".part", dst)                      # never leave a half-written image
                    break
                except Exception as e:
                    rz = None                                           # reconnect on the next attempt
                    if attempt == 3:
                        print("failed", src, repr(e), flush=True)
                    time.sleep(2 * (attempt + 1))
            with lock:
                progress["n"] += 1
                if progress["n"] % 500 == 0:
                    log(f"  images {progress['n']}/{total}")

    t0 = time.time()
    for rnd in range(3):                                                # up to 3 passes over anything still missing
        todo = missing()
        if not todo:
            break
        total = len(todo); progress["n"] = 0
        log(f"images to fetch (pass {rnd + 1}): {total}")
        n_workers = int(os.environ.get("N_WORKERS", 8))
        with ThreadPoolExecutor(n_workers) as ex:
            list(ex.map(worker, [todo[i::n_workers] for i in range(n_workers)]))
    left = missing()
    log(f"images done in {time.time()-t0:.0f}s; missing after retries: {len(left)}")
    if left:
        raise SystemExit(f"{len(left)} images could not be downloaded, e.g. {left[0][0]}. Re-run this cell to retry.")
    # MobileStereoNet file lists: one per sequence and GPU
    os.makedirs(f"{TMP}/stereo/lists", exist_ok=True)
    for s, frames in ref_of.items():
        names = sorted(frames)
        for g in range(2):
            with open(f"{TMP}/stereo/lists/{s}_gpu{g}.txt", "w") as fh:
                for t in names[g::2]:
                    fh.write(f"image_2/{t}.png image_3/{t}.png\n")


def phase_selectgroup():
    """Full mode: every labelled frame of the sequences in GROUP_SEQS (no sampling). Downloads the voxels zip."""
    seqs = os.environ["GROUP_SEQS"].split(",")
    zpath = wget(VOXELS_URL, f"{TMP}/voxels.zip")
    frames = set()
    for n in zipfile.ZipFile(zpath).namelist():
        m = VOX_RE.search(n)
        if m and m.group(3) == "label" and m.group(1) in seqs:
            frames.add((m.group(1), m.group(2)))
    train = sorted(f for f in frames if f[0] in TRAIN_SEQS)
    val = sorted(f for f in frames if f[0] == VAL_SEQ)
    os.makedirs(OUT, exist_ok=True)
    sel = {"train": [list(f) for f in train], "val": [list(f) for f in val],
           "summary": {"train": {"frames": len(train)}, "val": {"frames": len(val)}},
           "settings": {"mode": "full", "group_seqs": seqs}}
    json.dump(sel, open(f"{os.path.dirname(OUT)}/selection.json", "w"), indent=1)
    log(f"group {seqs}: {len(train)} train + {len(val)} val labelled frames")


if __name__ == "__main__":
    for phase in sys.argv[1:]:
        log(f"=== phase {phase} ===")
        globals()[f"phase_{phase}"]()

In [ ]:
%%writefile /kaggle/working/tools/depth_to_lidar.py
"""Depth maps (from MobileStereoNet) -> pseudo point clouds in MRA's format, cropped losslessly.

Same conversion as MRA's preprocess/utils/depth2lidar.py. Then, for each image frame, we keep only the points
that land inside MRA's scene box [0,51.2] x [-25.6,25.6] x [-2,4.4] (plus a 5 cm margin) after being moved into
the coordinate frame of the labelled frame that uses it, exactly as MRA's data loader does
(pts_ref = inv(pose_ref) @ pose_target @ pts). MRA's voxelizer discards every other point, so this changes nothing
the model sees. Points are saved in their own frame, as MRA expects, as x,y,z only (MRA's 4th column is always 1;
expand_subset.py adds it back before training), to fit Kaggle's 20 GB output limit.
"""
import os, sys, json, time, glob
import numpy as np
from multiprocessing import Pool

MRA = os.environ.get("MRA_DIR", "/kaggle/working/MRA_TIP")
OUT = os.environ.get("OUT_DIR", "/kaggle/working/kitti_subset/dataset")
TMP = os.environ.get("TMP_DIR", "/tmp/kitti_build")
sys.path.insert(0, f"{MRA}/preprocess/utils")
import kitti_util  # MRA's copy

LO = np.array([0.0, -25.6, -2.0]) - 0.05
HI = np.array([51.2, 25.6, 4.4]) + 0.05


def project_disp_to_depth(calib, depth, max_high):          # copied from MRA's depth2lidar.py
    rows, cols = depth.shape
    c, r = np.meshgrid(np.arange(cols), np.arange(rows))
    points = np.stack([c, r, depth]).reshape((3, -1)).T
    cloud = calib.project_image_to_velo(points)
    valid = (cloud[:, 0] >= 0) & (cloud[:, 2] < max_high)
    return cloud[valid]


def lidar_poses(seq):                                        # same as MRA's parse_poses
    Tr = np.identity(4)
    for line in open(f"{OUT}/sequences/{seq}/calib.txt"):
        if line.startswith("Tr:"):
            Tr[:3, :4] = np.array([float(x) for x in line.split(":", 1)[1].split()]).reshape(3, 4)
    Tr_inv = np.linalg.inv(Tr)
    poses = []
    for line in open(f"{OUT}/sequences/{seq}/poses.txt"):
        if not line.strip():
            continue
        v = [float(x) for x in line.split()]
        P = np.eye(4); P[:3, :4] = np.array(v[:12]).reshape(3, 4)
        poses.append(Tr_inv @ P @ Tr)
    return poses


def convert(args):
    seq, depth_file, ref_id, poses = args
    fid = os.path.basename(depth_file)[:-4]
    calib = kitti_util.Calibration(f"{OUT}/sequences/{seq}/calib.txt")
    cloud = project_disp_to_depth(calib, np.load(depth_file), 80)
    cloud = np.concatenate([cloud, np.ones((cloud.shape[0], 1))], 1).astype(np.float32)   # as MRA saves it
    T = np.linalg.inv(poses[int(ref_id)]) @ poses[int(fid)]
    in_ref = (T @ cloud.T).T[:, :3]                                                        # as MRA's loader does
    keep = np.all((in_ref > LO) & (in_ref < HI), axis=1)
    d = f"{OUT}/sequences_msnet3d_lidar/sequences/{seq}"
    os.makedirs(d, exist_ok=True)
    cloud[keep][:, :3].tofile(f"{d}/{fid}.bin")   # x,y,z only (4th column is always 1); expand_subset.py restores it
    os.remove(depth_file)
    return cloud.shape[0], int(keep.sum())


if __name__ == "__main__":
    ref_of = json.load(open(f"{TMP}/ref_of.json"))
    jobs = []
    for seq, frames in ref_of.items():
        poses = lidar_poses(seq)
        for f in sorted(glob.glob(f"{TMP}/depth/sequences/{seq}/*.npy")):
            fid = os.path.basename(f)[:-4]
            if fid in frames:
                jobs.append((seq, f, frames[fid], poses))
    t0 = time.time()
    with Pool(os.cpu_count()) as p:
        res = p.map(convert, jobs, chunksize=4)
    tot = sum(r[0] for r in res); kept = sum(r[1] for r in res)
    print(f"converted {len(res)} depth maps in {time.time()-t0:.0f}s; kept {kept/max(tot,1)*100:.1f}% of points "
          f"(avg {kept/max(len(res),1)/1e3:.0f}k points per frame)")
    missing = sum(len(v) for v in ref_of.values()) - len(res)
    print("frames without a depth map:", missing)

## C. Build and upload, group by group

In [ ]:
import subprocess, os, sys, json, glob, shutil, time
MRA, TMP = os.environ["MRA_DIR"], os.environ["TMP_DIR"]
PY_MRA = "/tmp/mra_env/bin/python"
MSNET = f"{MRA}/preprocess/mobilestereonet"
subprocess.run(f"{PY_MRA} -c 'import cv2' 2>/dev/null || {PY_MRA} -m pip -q install -c /tmp/constraints.txt "
               "opencv-python-headless==4.5.5.64", shell=True, check=True)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 1)

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def run(cmd, env=None, timeout=None, cwd=None):
    e = dict(os.environ, **(env or {}))
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, env=e, timeout=timeout, cwd=cwd)

def free_gb(path="/tmp"):
    return shutil.disk_usage(path).free / 1e9

def exists_on_kaggle(ref):
    r = subprocess.run(["kaggle", "datasets", "status", ref], capture_output=True, text=True)
    return r.returncode == 0 and "ready" in r.stdout.lower()

def baseline(seq):                                  # stereo baselines, as in building-subset
    return "388.1823" if seq in ("00", "01", "02") else "389.6304" if seq == "03" else "381.8293"

def depth(seq):
    lists = [f"{TMP}/stereo/lists/{seq}_gpu{g}.txt" for g in (0, 1)]
    lists = [l for l in lists if os.path.exists(l) and os.path.getsize(l) > 0]
    def cmd(lst):
        return [PY_MRA, "prediction.py", "--datapath", f"{TMP}/stereo/sequences/{seq}", "--testlist", lst,
                "--num_seq", seq, "--loadckpt", "./MSNet3D_SF_DS_KITTI2015.ckpt", "--dataset", "kitti",
                "--model", "MSNet3D", "--savepath", f"{TMP}/depth", "--baseline", baseline(seq)]
    t0 = time.time()
    if NGPU >= 2:
        procs = [subprocess.Popen(cmd(l), cwd=MSNET, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)),
                                  stdout=open(f"{TMP}/msnet_{seq}_{g}.log", "w"), stderr=subprocess.STDOUT)
                 for g, l in enumerate(lists)]
        codes = [p.wait(timeout=6 * 3600) for p in procs]
    else:
        codes = [subprocess.run(cmd(l), cwd=MSNET, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"),
                                stdout=open(f"{TMP}/msnet_{seq}_0.log", "a"), stderr=subprocess.STDOUT,
                                timeout=6 * 3600).returncode for l in lists]
    if any(codes):
        for f in glob.glob(f"{TMP}/msnet_{seq}_*.log"):
            print(open(f).read()[-3000:])
        raise RuntimeError(f"MobileStereoNet failed on sequence {seq}")
    shutil.rmtree(f"{TMP}/depth/disparity", ignore_errors=True)
    log(f"sequence {seq}: {len(glob.glob(f'{TMP}/depth/sequences/{seq}/*.npy'))} depth maps in {time.time()-t0:.0f}s")

for G in GROUPS:
    ref = f"{KAGGLE_USER}/{PREFIX}-{G.lower()}"
    if exists_on_kaggle(ref):
        log(f"=== group {G}: {ref} already exists on Kaggle - skipped"); continue
    seqs = GROUP_SEQS[G]
    root = f"/tmp/kfull_{G}"                         # build here; uploaded as kitti_full/ (merges with other groups)
    out = f"{root}/kitti_full/dataset"
    shutil.rmtree(root, ignore_errors=True); shutil.rmtree(f"{TMP}/stereo", ignore_errors=True)
    shutil.rmtree(f"{TMP}/depth", ignore_errors=True)
    log(f"=== group {G} {seqs} -> {ref} | free disk {free_gb():.0f} GB")
    env = {"OUT_DIR": out, "GROUP_SEQS": ",".join(seqs)}
    run([sys.executable, "/kaggle/working/tools/build_full.py", "selectgroup", "meta", "labels"], env=env, timeout=3 * 3600)
    for s in seqs:
        log(f"--- sequence {s}: images | free disk {free_gb():.0f} GB")
        run([sys.executable, "/kaggle/working/tools/build_full.py", "images"], env=dict(env, ONLY_SEQ=s), timeout=5 * 3600)
        depth(s)
        run([sys.executable, "/kaggle/working/tools/depth_to_lidar.py"], env=env, timeout=3 * 3600)
        ref_of = json.load(open(f"{TMP}/ref_of.json"))[s]
        n_img = len(glob.glob(f"{out}/sequences/{s}/image_2/*.png"))
        n_pc = len(glob.glob(f"{out}/sequences_msnet3d_lidar/sequences/{s}/*.bin"))
        assert n_img >= len(ref_of) and n_pc >= len(ref_of), f"sequence {s}: {n_img} images, {n_pc} clouds, need {len(ref_of)}"
        shutil.rmtree(f"{TMP}/stereo", ignore_errors=True); shutil.rmtree(f"{TMP}/depth", ignore_errors=True)
        log(f"sequence {s} done: {n_img} images, {n_pc} point clouds")
    sizes = subprocess.run(f"du -sh {out}/sequences {out}/labels {out}/sequences_msnet3d_lidar", shell=True,
                           capture_output=True, text=True).stdout
    print(sizes)
    up = f"/tmp/upload_{G}"; shutil.rmtree(up, ignore_errors=True); os.makedirs(up)
    log("packing (tar removes the files as it goes, so disk use does not double)")
    run(f"cd {root} && tar cf {up}/kitti_full_{G.lower()}.tar --remove-files kitti_full", timeout=3 * 3600)
    json.dump({"title": f"SemanticKITTI MRA full {G}", "id": ref, "licenses": [{"name": "other"}],
               "description": f"SemanticKITTI (KITTI odometry) sequences {', '.join(seqs)} in MRA's compact format with "
                              "MobileStereoNet pseudo-LiDAR. Built by kaggle/build_full_dataset.ipynb. KITTI and "
                              "SemanticKITTI licences apply (CC BY-NC-SA)."},
              open(f"{up}/dataset-metadata.json", "w"), indent=1)
    log(f"uploading {ref} ({os.path.getsize(f'{up}/kitti_full_{G.lower()}.tar')/1e9:.1f} GB, private)")
    run(["kaggle", "datasets", "create", "-p", up, "-q"], timeout=6 * 3600)
    shutil.rmtree(up, ignore_errors=True); shutil.rmtree(root, ignore_errors=True)
    log(f"UPLOADED {ref}")
log("all requested groups done")

## D. Use it on molab

When all four groups show up as **ready** on Kaggle (processing a large dataset can take a while after the upload),
set in the molab notebook:

```python
DATA = "kaggle://datasets/daniyalimran/semantickitti-mra-full-d,kaggle://datasets/daniyalimran/semantickitti-mra-full-a,kaggle://datasets/daniyalimran/semantickitti-mra-full-b,kaggle://datasets/daniyalimran/semantickitti-mra-full-c"
```

`molab/prepare_data.py` downloads every listed dataset into the same folder (they merge into one `kitti_full/` tree),
rebuilds MRA's format and checks every file. Only group D -> the full validation set (sequence 08) plus sequence 09.